In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("darkgrid")

### Sütun isimlerini atama ve RUL hesaplaması yapma

In [ ]:
columns = ["engine_id", "time_cycles", "opr_setting_1", "opr_setting_2", "opr_setting_3"] + [f's{i}' for i in range(1,22)]

train_df = pd.read_csv("train_FD001.txt", sep=' ', header=None, names=columns)

# Her bir motorun ulaştığı maksimum döngü sayısını buluyoruz
max_cycles = train_df.groupby('engine_id')['time_cycles'].max().reset_index()
max_cycles.columns = ['engine_id', 'max_cycle']

train_df = pd.merge(train_df, max_cycles, on='engine_id', how='left')

# RUL hesaplaması: Maksimum döngü - Mevcut döngü
train_df['RUL'] = train_df['max_cycle'] - train_df['time_cycles']

train_df = train_df.drop('max_cycle', axis=1)

print(train_df[['engine_id', 'time_cycles', 'RUL']].head(10))


   engine_id  time_cycles     RUL
0    -0.0007      -0.0004  0.0010
1     0.0019      -0.0003  0.0009
2    -0.0043       0.0003  0.0002
3     0.0007       0.0000  0.0005
4    -0.0019      -0.0002  0.0007
5    -0.0043      -0.0001  0.0006
6     0.0010       0.0001  0.0004
7    -0.0034       0.0003  0.0003
8     0.0008       0.0001  0.0005
9    -0.0033       0.0001  0.0004


### Sabit sensör değerlerini temizleme ve normalize etme

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Varyansı sıfır olan sütunları at (bilgi taşımaz)
sensor_cols = [f's{i}' for i in range(1,22)]
low_var = train_df[sensor_cols].std() < 0.01
drop_cols = low_var[low_var].index.tolist()
train_df.drop(columns=drop_cols, inplace=True)

useful_sensors = [c for c in sensor_cols if c not in drop_cols]

scaler = MinMaxScaler()
train_df[useful_sensors] = scaler.fit_transform(train_df[useful_sensors])